# Mikado Judge — YOLO-OBB Training (Google Colab)

This notebook trains a YOLO-OBB model to detect Mikado sticks.  
**Runtime → Change runtime type → T4 GPU** before running.

## Workflow
1. Mount Google Drive and set paths
2. Install dependencies (Ultralytics + Blender)
3. Clone / pull repositories (mikado-judge + mikado-blender)
4. Extract real dataset + generate synthetic data
5. Tile all images into 1280x1280 crops
6. Train the model
7. Evaluate and save weights to Drive

## 1. Setup — Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURE THESE to match your Google Drive layout
# ---------------------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/Data/mikado-judge'

DATASET_ZIP = f'{DRIVE_ROOT}/dataset.tar.gz'
DATASET_DIR = f'{DRIVE_ROOT}/datasets/mikado'   # original high-res images
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

# ---------------------------------------------------------------------------
# Synthetic data
# ---------------------------------------------------------------------------
SYNTHETIC_COUNT = 200       # number of synthetic images to generate (0 = skip)
SYNTHETIC_STYLE = 'realistic'  # 'simple' or 'realistic' (wood grain + paint tips)

# ---------------------------------------------------------------------------
# Model — yolo11m-obb has 3.3x more capacity than yolo11s for thin features
# ---------------------------------------------------------------------------
BASE_MODEL = 'yolo11m-obb.pt'
RUN_NAME = 'mikado_v3'

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
# With tiling, input images are 1280x1280 crops at native resolution.
# Sticks are ~28px wide (vs 8px without tiling). batch=4 fits on T4.
EPOCHS = 200
IMGSZ = 1280
BATCH = 4              # tiles are 1280x1280 — lighter than full 4624px images

import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Base model:       {BASE_MODEL}')
print(f'Image size:       {IMGSZ}')
print(f'Batch size:       {BATCH}')
print(f'Epochs:           {EPOCHS}')
print(f'Synthetic count:  {SYNTHETIC_COUNT}')
print(f'Synthetic style:  {SYNTHETIC_STYLE}')

In [ ]:
## Extract dataset archive (skips if already extracted)
import os, tarfile, zipfile
from pathlib import Path

def _extract(archive_path, dest_parent):
    """Extract .zip or .tar.gz archive into dest_parent."""
    p = str(archive_path)
    if p.endswith('.zip'):
        with zipfile.ZipFile(p, 'r') as f:
            f.extractall(dest_parent)
    else:  # .tar.gz / .tgz
        with tarfile.open(p, 'r:gz') as f:
            f.extractall(dest_parent)

if Path(DATASET_DIR).exists() and any(Path(DATASET_DIR).iterdir()):
    print(f'Dataset already extracted at {DATASET_DIR}, skipping.')
else:
    print(f'Extracting {DATASET_ZIP} ...')
    parent = str(Path(DATASET_DIR).parent)
    os.makedirs(parent, exist_ok=True)
    _extract(DATASET_ZIP, parent)
    # Archive root folder is called 'dataset'; rename to match DATASET_DIR
    extracted_root = os.path.join(parent, 'dataset')
    if os.path.exists(extracted_root) and extracted_root != DATASET_DIR:
        os.rename(extracted_root, DATASET_DIR)
    imgs = list(Path(DATASET_DIR).rglob('*.jpg')) + list(Path(DATASET_DIR).rglob('*.png'))
    print(f'Extracted to {DATASET_DIR} — {len(imgs)} images found.')

## 2. Install dependencies (Ultralytics + Blender)

In [ ]:
!pip install ultralytics --quiet
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Install Blender for synthetic data generation
if SYNTHETIC_COUNT > 0:
    !apt-get install -y blender 2>&1 | tail -3
    !apt-get install -y python3-yaml 2>&1 | tail -1
    !/usr/bin/blender --version 2>&1 | head -1

    # Cache Cycles CUDA kernels to Drive (compiles once ~5min, loads from cache after)
    import os
    KERNEL_CACHE_DIR = f'{DRIVE_ROOT}/blender_cycles_cache'
    os.makedirs(KERNEL_CACHE_DIR, exist_ok=True)
    os.environ['CYCLES_KERNEL_PATH'] = KERNEL_CACHE_DIR
    cache_files = list(__import__('pathlib').Path(KERNEL_CACHE_DIR).glob('*.cubin'))
    print(f'Cycles kernel cache: {len(cache_files)} cached kernels')
else:
    print('Skipping Blender install (SYNTHETIC_COUNT=0)')

## 3. Clone / update repositories

In [ ]:
REPO_DIR = '/content/mikado-judge'
REPO_URL = 'https://github.com/spyszkowski/mikado-judge.git'

BLENDER_REPO_DIR = '/content/mikado-blender'
BLENDER_REPO_URL = 'https://github.com/spyszkowski/mikado-blender.git'

import os

# Clone/update mikado-judge
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!pip install -e {REPO_DIR} --quiet

# Clone/update mikado-blender (for synthetic data)
if SYNTHETIC_COUNT > 0:
    if os.path.exists(BLENDER_REPO_DIR):
        !cd {BLENDER_REPO_DIR} && git pull
    else:
        !git clone --branch master {BLENDER_REPO_URL} {BLENDER_REPO_DIR}

print('Repositories ready.')

## 4. Verify dataset

In [ ]:
import os
from pathlib import Path

dataset_path = Path(DATASET_DIR)
train_images = list((dataset_path / 'images' / 'train').glob('*'))
val_images = list((dataset_path / 'images' / 'val').glob('*'))
train_labels = list((dataset_path / 'labels' / 'train').glob('*.txt'))
val_labels = list((dataset_path / 'labels' / 'val').glob('*.txt'))

print(f'Train images: {len(train_images)}')
print(f'Train labels: {len(train_labels)}')
print(f'Val images:   {len(val_images)}')
print(f'Val labels:   {len(val_labels)}')

if len(train_images) == 0:
    print('\n⚠ WARNING: No training images found. Upload your dataset to Drive first.')
else:
    print('\n✓ Dataset looks good.')

In [ ]:
# Write a dataset.yaml pointing to the Colab paths
# NOTE: class 0 (mikado) has 0 annotations — model won't learn it but
# keeping 5 classes avoids relabeling. Cost: 1 wasted output channel.
import yaml

dataset_yaml_path = f'{DATASET_DIR}/dataset.yaml'
dataset_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 5,
    'names': {0: 'mikado', 1: 'blue', 2: 'red', 3: 'yellow', 4: 'green'},
}
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)
print(f'Written: {dataset_yaml_path}')

# Save original dataset dir before tiling overrides it
ORIGINAL_DATASET_DIR = DATASET_DIR

## 4b. Generate synthetic training data (optional)

Uses Blender to render synthetic stick piles with physics simulation.
Synthetic images are 1280x960 (already tile-sized) and are merged directly
into the tiled dataset in the next step. Set `SYNTHETIC_COUNT=0` to skip.

In [ ]:
# Generate synthetic data using Blender (skipped if SYNTHETIC_COUNT=0)
import subprocess, os
from pathlib import Path

SYNTHETIC_DIR = f'{DRIVE_ROOT}/synthetic'

if SYNTHETIC_COUNT == 0:
    print('Synthetic generation skipped (SYNTHETIC_COUNT=0).')
elif Path(SYNTHETIC_DIR, 'images').exists() and \
     len(list(Path(SYNTHETIC_DIR, 'images').glob('*'))) >= SYNTHETIC_COUNT:
    n = len(list(Path(SYNTHETIC_DIR, 'images').glob('*')))
    print(f'Synthetic data already exists: {n} images. Delete {SYNTHETIC_DIR} to regenerate.')
else:
    print(f'Generating {SYNTHETIC_COUNT} synthetic images at 4624x3472 (highres for tiling)...')
    os.makedirs(SYNTHETIC_DIR, exist_ok=True)

    env = os.environ.copy()
    env['CYCLES_KERNEL_PATH'] = f'{DRIVE_ROOT}/blender_cycles_cache'

    cmd = [
        '/usr/bin/blender', '--background',
        '--python', f'{BLENDER_REPO_DIR}/scripts/generate.py', '--',
        '--count', str(SYNTHETIC_COUNT),
        '--output', SYNTHETIC_DIR,
        '--config', f'{BLENDER_REPO_DIR}/configs',
        '--stick-style', SYNTHETIC_STYLE,
        '--resolution', 'highres',  # 4624x3472 — will be tiled like real data
    ]

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if result.stdout:
        # Show progress lines
        for line in result.stdout.splitlines():
            if line.startswith('[') or 'Done' in line:
                print(line)
    if result.returncode != 0:
        print(f'ERROR (exit {result.returncode}):')
        print(result.stderr[-2000:])
    else:
        n = len(list(Path(SYNTHETIC_DIR, 'images').glob('*')))
        print(f'Generated {n} synthetic images.')

## 4c. Tile all images (real + synthetic)

Both real and synthetic images are 4624x3472. Tiling into 1280x1280 crops
keeps sticks at native resolution (~24-28px wide).

```
68 real images x ~20 tiles     = ~1360 tiles  (sticks 28px wide)
200 synthetic  x ~20 tiles     = ~4000 tiles  (sticks 24px wide)
Total: ~5360 training images
```

In [ ]:
# Tile real + synthetic images into 1280x1280 crops
from pathlib import Path

TILED_DIR = f'{DRIVE_ROOT}/datasets/mikado_tiled'

# --- Step 1: Tile real images ---
if Path(TILED_DIR).exists() and any((Path(TILED_DIR) / 'images' / 'train').glob('*')):
    n_existing = len(list((Path(TILED_DIR) / 'images' / 'train').glob('*')))
    print(f'Tiled dataset already exists: {n_existing} train tiles. Skipping.')
    print('Delete the tiled directory to force re-tiling:')
    print(f'  !rm -rf {TILED_DIR}')
else:
    print('Tiling real dataset...')
    !python {REPO_DIR}/scripts/tile_dataset.py \
        --images {ORIGINAL_DATASET_DIR}/images/train \
        --labels {ORIGINAL_DATASET_DIR}/labels/train \
        --output {TILED_DIR} \
        --val-images {ORIGINAL_DATASET_DIR}/images/val \
        --val-labels {ORIGINAL_DATASET_DIR}/labels/val \
        --crop-size 1280 \
        --overlap 0.3 \
        --min-stick-length 0.3 \
        --classes {REPO_DIR}/configs/classes.yaml

    # --- Step 2: Tile synthetic images (same pipeline) ---
    synth_dir = Path(f'{DRIVE_ROOT}/synthetic')
    if SYNTHETIC_COUNT > 0 and (synth_dir / 'images').exists():
        SYNTH_TILED = '/content/synthetic_tiled'
        print('\nTiling synthetic images...')
        !python {REPO_DIR}/scripts/tile_dataset.py \
            --images {synth_dir}/images \
            --labels {synth_dir}/labels \
            --output {SYNTH_TILED} \
            --crop-size 1280 \
            --overlap 0.3 \
            --min-stick-length 0.3 \
            --classes {REPO_DIR}/configs/classes.yaml

        # Merge synthetic tiles into tiled dataset (train only)
        import shutil
        train_img = Path(TILED_DIR) / 'images' / 'train'
        train_lbl = Path(TILED_DIR) / 'labels' / 'train'
        synth_img = Path(SYNTH_TILED) / 'images' / 'train'
        synth_lbl = Path(SYNTH_TILED) / 'labels' / 'train'
        if synth_img.exists():
            copied = 0
            for img in synth_img.glob('*'):
                lbl = synth_lbl / (img.stem + '.txt')
                shutil.copy2(img, train_img / img.name)
                if lbl.exists():
                    shutil.copy2(lbl, train_lbl / lbl.name)
                copied += 1
            print(f'Merged {copied} synthetic tiles into training set.')

# --- Stats ---
n_train = len(list((Path(TILED_DIR) / 'images' / 'train').glob('*')))
n_val = len(list((Path(TILED_DIR) / 'images' / 'val').glob('*')))
print(f'\nFinal dataset: {TILED_DIR}')
print(f'  Train: {n_train} tiles')
print(f'  Val:   {n_val} tiles (real only)')

# --- Write dataset.yaml for tiled dataset ---
import yaml
dataset_yaml_path = f'{TILED_DIR}/dataset.yaml'
dataset_yaml = {
    'path': TILED_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 5,
    'names': {0: 'mikado', 1: 'blue', 2: 'red', 3: 'yellow', 4: 'green'},
}
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

# Point training at the tiled dataset
DATASET_DIR = TILED_DIR
print(f'Training dataset: {DATASET_DIR}')

## 5. Train the model

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=dataset_yaml_path,
    task='obb',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=RUN_NAME,
    project='/content/runs/obb',
    exist_ok=True,

    # === Loss weights — tuned for thin elongated objects ===
    box=10.0,            # +33% over default — precise OBB localization
    cls=0.3,             # -40% — tips are tiny, classification is hard
    dfl=2.0,             # +33% — sharper box edges

    # === Optimizer ===
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.1,
    cos_lr=True,
    warmup_epochs=5,
    weight_decay=0.001,

    # === Early stopping ===
    patience=30,
    save_period=10,

    # === Augmentation ===
    # With tiling, sticks are 28px wide — can handle more augmentation.
    degrees=180,         # full rotation — sticks at all angles
    mosaic=0.8,          # high but not 1.0 — some clean samples help
    close_mosaic=20,     # disable mosaic for last 20 epochs
    copy_paste=0.3,      # overlapping sticks augmentation
    mixup=0.0,           # disabled — overlapping piles = noise
    erasing=0.1,         # very low — higher values erase thin sticks entirely
    shear=0.0,           # no shear — distorts OBBs
    scale=0.5,           # safe now — sticks stay >14px at 0.5 scale
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,
    flipud=0.5,

    # === Advanced ===
    amp=True,
    dropout=0.1,
    cache='ram',
    pretrained=True,
    verbose=True,
)

## 5b. Resume interrupted training (run this instead of cell 5 if training was interrupted)

In [ ]:
# Uncomment and set the path to resume training
# RESUME_WEIGHTS = f'/content/runs/obb/{RUN_NAME}/weights/last.pt'
# from ultralytics import YOLO
# model = YOLO(RESUME_WEIGHTS)
# model.train(resume=True)

## 6. Evaluate the trained model

In [ ]:
best_weights = f'/content/runs/obb/{RUN_NAME}/weights/best.pt'
model = YOLO(best_weights)
metrics = model.val(data=dataset_yaml_path, task='obb', imgsz=IMGSZ)
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 7. Visualise sample predictions

In [ ]:
import random
from pathlib import Path
from IPython.display import display
from PIL import Image

best_weights = f'/content/runs/obb/{RUN_NAME}/weights/best.pt'
model = YOLO(best_weights)

val_imgs = list((Path(DATASET_DIR) / 'images' / 'val').glob('*'))
samples = random.sample(val_imgs, min(4, len(val_imgs)))

for img_path in samples:
    # augment=True enables test-time augmentation (TTA) for ~2-5% mAP boost
    result = model.predict(str(img_path), imgsz=IMGSZ, conf=0.1, augment=True, verbose=False)[0]
    vis = result.plot(line_width=4)   # thicker lines — sticks are thin
    # Downscale for display (original images are ~4600px wide)
    img = Image.fromarray(vis[:, :, ::-1])
    w, h = img.size
    img = img.resize((min(w, 1200), int(h * min(w, 1200) / w)), Image.LANCZOS)
    n = len(result.obb) if result.obb is not None else 0
    print(f'{img_path.name}: {n} detections')
    display(img)

## 8. Save best weights to Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{WEIGHTS_DIR}/{RUN_NAME}_{timestamp}_best.pt'
shutil.copy2(best_weights, dest)
print(f'Saved weights → {dest}')

# Also copy training curves
results_csv = f'/content/runs/obb/{RUN_NAME}/results.csv'
if os.path.exists(results_csv):
    shutil.copy2(results_csv, f'{WEIGHTS_DIR}/{RUN_NAME}_{timestamp}_results.csv')
    print('Saved training results CSV.')